# Qdrant Vector Store With llama-index-pydocker

Qdrant is a dedicated vector database with filtering, payload storage, and hybrid search built in. Running it locally requires pulling a Docker image, exposing the right port, and waiting for the HTTP API to be ready before any client code can run.

This notebook shows how `llama-index-pydocker` handles all of that with one import swap, then demonstrates indexing, retrieval, and automatic teardown.

## Table Of Contents

- [Prerequisites](#prerequisites)
- [1. Start Qdrant With One Import Swap](#1-start-qdrant-with-one-import-swap)
- [2. Index Documents For Retrieval](#2-index-documents-for-retrieval)
- [3. Run And Validate Retrieval](#3-run-and-validate-retrieval)
- [4. Context Manager Teardown](#4-context-manager-teardown)
- [5. Remote Passthrough (No Docker)](#5-remote-passthrough-no-docker)
- [6. Conclusion](#6-conclusion)

## Prerequisites

- Docker Desktop must be running.
- Install dependencies:
  ```bash
  pip install "llama-index-pydocker[qdrant]"
  ```
- No environment variables required.

## 1. Start Qdrant With One Import Swap

Change the import from `llama_index.vector_stores.qdrant` to `llama_index_pydocker`. The Qdrant container is started automatically when the URL resolves to localhost.

In [ ]:
import sys
import uuid
import tempfile
from pathlib import Path

sys.path.insert(0, str(Path().cwd().parent))

from llama_index_pydocker import QdrantVectorStore
from docker_db import QdrantConfig

In [ ]:
temp_dir = Path(tempfile.mkdtemp())
container_name = f"demo-qdrant-{uuid.uuid4().hex[:8]}"

cfg = QdrantConfig(
    database="demo_collection",
    project_name="demo",
    container_name=container_name,
    volume_path=temp_dir / "qdrantdata",
    vector_size=384,
    retries=20,
    delay=2,
)

store = QdrantVectorStore(
    collection_name="demo_collection",
    url=f"http://localhost:{cfg.port}",
    docker_config=cfg,
)

print(f"Qdrant started:  {container_name}")
print(f"Collection:      demo_collection")

## 2. Index Documents For Retrieval

Pass `store` to `StorageContext`. LlamaIndex inserts vectors and payloads into the Qdrant collection.

In [ ]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.embeddings import MockEmbedding
from llama_index.core.schema import Document

documents = [
    Document(text="Qdrant is a vector similarity search engine built in Rust."),
    Document(text="Payload filtering in Qdrant allows structured and semantic search to be combined."),
    Document(text="Docker makes local infrastructure reproducible for development and testing."),
]

embed_model = MockEmbedding(embed_dim=384)
storage_context = StorageContext.from_defaults(vector_store=store)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=False,
)

print(f"Indexed {len(documents)} documents.")

## 3. Run And Validate Retrieval

Retrieve nodes from Qdrant to confirm the full data path is working.

In [ ]:
query = "What language is Qdrant built in?"
retriever = index.as_retriever(similarity_top_k=2)
results = retriever.retrieve(query)

print(f"Retrieved {len(results)} nodes")
for i, node in enumerate(results, start=1):
    print(f"\nResult {i} (score={node.score:.4f})")
    print(node.text)

assert len(results) > 0

## 4. Context Manager Teardown

Call `store.stop()` to remove the container, or use `with` so teardown happens automatically.

In [ ]:
store.stop()
print(f"Container '{container_name}' removed.")

In [ ]:
# Equivalent pattern with automatic teardown
new_name = f"demo-qdrant-{uuid.uuid4().hex[:8]}"
new_cfg = cfg.model_copy(update={"container_name": new_name,
                                   "volume_path": temp_dir / new_name})

with QdrantVectorStore(
    collection_name="demo_collection",
    url=f"http://localhost:{new_cfg.port}",
    docker_config=new_cfg,
) as s:
    print(f"Container up: {s._db.config.container_name}")
print("Container removed.")

## 5. Remote Passthrough (No Docker)

Point the store at a Qdrant Cloud cluster and the wrapper is completely transparent.

In [ ]:
# Uncomment to target a hosted Qdrant cluster — Docker is never started
#
# store = QdrantVectorStore(
#     collection_name="docs",
#     url="https://my-cluster.qdrant.io:6333",
#     api_key="sk-...",
# )
# assert store._db is None   # no container

print("Remote passthrough: Docker is never touched for non-localhost URLs.")

## 6. Conclusion

You have completed the workflow introduced at the top:
1. Started a Qdrant container with a single import swap.
2. Indexed documents through `QdrantVectorStore` without managing a client manually.
3. Retrieved nodes and confirmed the data path is working end to end.
4. Cleaned up with `stop()` and with a context manager.

Key next steps:
- Replace `MockEmbedding` with a real embedding model and update `vector_size` in `QdrantConfig`.
- Use `QdrantConfig.database` to name the collection and `volume_path` to persist data across runs.
- Switch `url` to a Qdrant Cloud endpoint to deploy without any other code changes.